### Lesson 4: Persistence and Streaming

In [1]:
# !pip install langgraph.checkpoint.sqlite
# !pip install aiosqlite

In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv('GEMINI_API_KEY')
tavily_api_key = os.getenv('TAVILY_API_KEY')

In [40]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
# Initialize the Gemini chat model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # You can choose other Gemini models like "gemini-pro"
    temperature=0.7,
    max_output_tokens=2048,
    api_key=gemini_api_key
)

E0000 00:00:1759860906.731990 10751395 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [5]:
tool = TavilySearchResults(max_results=2)

/var/folders/70/3n6r904d4zb5zh68lfyslrt80000gn/T/ipykernel_33396/4289725543.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


In [6]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [30]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
db_path = 'checkpoints.db'
conn = sqlite3.connect(db_path, check_same_thread=False)

memory = SqliteSaver(conn)

In [31]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
    
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0
    
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:
                print("\n ....bad tool name....")
                result = "bad tool name, retry"
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [32]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that! 
Use TavilySearchResults tool for getting the information for an external source.
"""

abot = Agent(llm, [tool], checkpointer=memory, system=prompt)

In [33]:
messages = [HumanMessage(content="What is the weather forecast in sf?")]
thread = {"configurable": {"thread_id": "1"}}

In [34]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='The weather forecast for San Francisco in October 2025 predicts temperatures around 63-64°F, with mostly sunny days. There might be a few rainy days, but generally not more than three.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--75b85e4f-03d7-4731-a8f9-a86de653b521-0', usage_metadata={'input_tokens': 10321, 'output_tokens': 78, 'total_tokens': 10399, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 33}})]


In [35]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='The weather forecast for Los Angeles in October 2025 predicts highs generally in the low to high 70s°F. You can expect mostly comfortable weather with a few rainy days, but typically not more than one.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--467ad2d8-a6eb-42b5-b386-af36753e7dcd-0', usage_metadata={'input_tokens': 10373, 'output_tokens': 105, 'total_tokens': 10478, 'input_token_details': {'cache_read': 9695}, 'output_token_details': {'reasoning': 58}})]}


In [36]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Based on the forecasts for October 2025, Los Angeles is warmer than San Francisco. San Francisco is forecast to have temperatures around 63-64°F, while Los Angeles is forecast to have highs generally in the low to high 70s°F.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--1f73e5f0-661c-43db-b377-4c0e20044af0-0', usage_metadata={'input_tokens': 10427, 'output_tokens': 109, 'total_tokens': 10536, 'input_token_details': {'cache_read': 9691}, 'output_token_details': {'reasoning': 51}})]}


In [37]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='I cannot tell you which one is warmer without knowing what items you would like to compare. Please provide me with the items.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--91a02656-f119-47d1-9877-e73a4bd9a521-0', usage_metadata={'input_tokens': 205, 'output_tokens': 60, 'total_tokens': 265, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 35}})]}


In [38]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

async with AsyncSqliteSaver.from_conn_string(':memory:') as checkpointer:
    abot = Agent(llm, [tool], system=prompt, checkpointer=checkpointer)
    messages = [HumanMessage(content='What is the weather forecast in SF?')]
    thread = {'configurable': {'thread_id': '4'}}
    async for event in abot.graph.astream_events({'messages': messages}, thread, version='v1'):
        kind = event['event']
        if kind == 'on_chat_model_stream':
            content = event['data']['chunk'].content
            if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
                print(content, end='|')

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather forecast San Francisco'}, 'id': 'c108667f-b2e5-4ff9-9306-48653f51f7ee', 'type': 'tool_call'}
Back to the model!
